# MovingWindow

`MovingWindow` slides one or more fixed-length windows across the series, scores each split-in-the-middle interval with a `change_score`, and picks the local maxima that exceed the penalty. Runtime is linear in the series length, which makes it a good default when you already have a rough idea of the minimum separation between two changepoints.

## Single bandwidth

The example below detects mean shifts with the [CUSUM](../../api_reference/auto_generated/skchange.interval_scorers.CUSUM.rst) statistic and a single bandwidth of 20 samples on either side of every candidate changepoint.

In [ ]:
import plotly.io as pio

from skchange.datasets import generate_piecewise_normal_data
from skchange.detectors import MovingWindow
from skchange.interval_scorers import CUSUM
from skchange.utils.plotting import plot_detections

pio.renderers.default = "notebook"

X = generate_piecewise_normal_data(
    means=[0, 10, 0, -3, 5, 1],
    lengths=[30, 5, 15, 50, 60, 40],
    seed=0,
)

detector = MovingWindow(CUSUM(), bandwidth=20, penalty_scale=1.0)
changepoints = detector.fit_predict(X)

plot_detections(X, changepoints=changepoints).show()
print(changepoints)

The short spike segment `[30, 35)` is unlikely to be picked up at this bandwidth. Passing an array of bandwidths, or reducing `min_bandwidth`, lets `MovingWindow` react to shorter segments too.

## Multiple bandwidths

Passing an array of bandwidths runs the moving-window scan once per bandwidth and combines the results. This lets `MovingWindow` react to both short and long segments in the same series without having to pick a single bandwidth up-front, but increases run-time proportionally to the number of bandwidths.

In [ ]:
detector = MovingWindow(
    CUSUM(),
    bandwidth=[3, 10, 30],
    penalty_scale=1.0,
)
changepoints = detector.fit_predict(X)

plot_detections(X, changepoints=changepoints).show()
print(changepoints)

## Parameters worth knowing

- `change_score`: Any interval scorer of type `change_score`.
- `penalty`, `penalty_scale`, `agg`: Same role as in [SeededBinarySegmentation](seeded_binseg.ipynb).
- `bandwidth`: A single integer, an array of integers (multi-scale), or `None` (a default derived from `min_bandwidth`). Directly controls what segment lengths the detector is sensitive to.
- `min_bandwidth`: Lower bound on the window half-width.
- `selection_method`: How competing candidate changepoints are resolved. Defaults to `"local_optimum"`.

See the full API reference for [MovingWindow](../../api_reference/auto_generated/skchange.detectors.MovingWindow.rst).

## See also

- [SeededBinarySegmentation](seeded_binseg.ipynb): Explores many scales automatically at log-linear cost.
- [PELT](pelt.ipynb): Exact segmentation for cost-based scorers.